# 🎬 Geração de Vídeo com CogVideoX-5B-I2V

Este notebook gera vídeos cinematográficos a partir de uma imagem e um prompt descritivo, utilizando o modelo **CogVideoX-5B-I2V**.

✅ Compatível com Google Colab + GPU T4
✅ Suporta imagem de entrada + texto
✅ Interface com Gradio para facilidade de uso


In [ ]:
# ⚙️ Instalar dependências
!pip install -q git+https://github.com/THUDM/CogVideoX.git@main
!pip install -q gradio einops decord transformers accelerate safetensors

In [ ]:
# 🔑 Login Hugging Face (obrigatório para carregar o modelo)
from huggingface_hub import login
login()  # Cole aqui seu token Hugging Face

In [ ]:
# 📦 Importações principais
import os
import torch
from PIL import Image
import gradio as gr
from cogvideox.models import CogVideoX
from cogvideox.utils.video_utils import save_video
from torchvision import transforms

os.makedirs("outputs", exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# 🚀 Carregar modelo CogVideoX
pipe = CogVideoX.from_pretrained("THUDM/cogvideox-5b-i2v", torch_dtype=torch.float16).to(device)

In [ ]:
# 🧠 Função de geração de vídeo
def generate_video(image_path, prompt):
    image = Image.open(image_path).convert("RGB")
    preprocess = transforms.Compose([
        transforms.Resize((512, 512)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ])
    image_tensor = preprocess(image).unsqueeze(0).to(device)

    output = pipe(
        prompt=prompt,
        input_image=image_tensor,
        num_frames=24,
        decode_chunk_size=8,
        output_type="pil"
    )

    frames = output["video"]
    output_path = f"outputs/generated_video.mp4"
    save_video(frames, output_path, fps=8)
    return output_path

In [ ]:
# 🎛️ Interface Gradio
def run_app():
    with gr.Blocks() as demo:
        gr.Markdown("## 🎬 CogVideoX: Geração de vídeo a partir de imagem + descrição")
        with gr.Row():
            img = gr.Image(type="filepath", label="Imagem de Entrada")
            txt = gr.Textbox(label="Prompt Descritivo", value="A cinematic scene of waves crashing under sunset.")
        btn = gr.Button("🎥 Gerar Vídeo")
        vid = gr.Video(label="🎞️ Resultado")
        btn.click(fn=generate_video, inputs=[img, txt], outputs=vid)
    demo.launch(share=True)

run_app()